# Handwritten circuit component detection (Digitize-HCD)
YOLO11m fine-tuned at 1024 px. **Task:** detect every component in a hand-drawn circuit diagram and classify it (17 classes).

**How to run (Kaggle):** attach the Digitize-HCD dataset, Accelerator = GPU T4 (the notebook uses one GPU, `device=0`), Internet = On (needed to install `ultralytics` and download the `yolo11m.pt` weights).

**`RUN_TEST` flag:** keep `RUN_TEST = False` while developing and set it to `True` only for the final run, so the test set is evaluated once.

| Cell | Content |
|---|---|
| 1 | Setup |
| 2 | Load and inspect the annotations |
| 3 | Clean, split and convert to YOLO format |
| 4 | Visual check of the conversion |
| 5 | Train |
| 6 | Evaluate on validation (also picks the confidence threshold) |
| 7 | Final evaluation on test |
| 8 | Error analysis |
| 9 | Save outputs |
| 10 | Test on my own image |



## 1. Setup

In [ ]:
!pip install -q ultralytics==8.4.182
import os, json, shutil, random, glob, datetime
import numpy as np, pandas as pd, cv2, torch, yaml
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

SEED = 42
RUN_TEST = False                     # False while developing, True only for the final run
random.seed(SEED); np.random.seed(SEED)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

ANN = glob.glob("/kaggle/input/**/component_annotations.json", recursive=True)[0]
BASE = os.path.dirname(ANN)
IMG_DIR = os.path.join(BASE, "Circuit Diagram Images")
OUT  = "/kaggle/working/hcd_yolo"    # converted YOLO dataset
RUNS = "/kaggle/working/runs"        # training / eval outputs
DATA_YAML = f"{OUT}/data.yaml"
print(BASE)

## 2. Load and inspect the annotations
Annotations are in COCO format: each box is `[x, y, w, h]` in pixels (top-left corner, width, height).

- The per-class counts show the **class imbalance**, which is why the split in cell 3 is stratified.
- The box sizes justify `imgsz=1024`: YOLO resizes the longest side of each image to `imgsz`, so we compute how large the *short side* of each box becomes. At 640 px many symbols get very small; at 1024 px they stay larger.

In [ ]:
coco = json.load(open(ANN))
cats = {c["id"]: c["name"] for c in coco["categories"]}
imgs = pd.DataFrame(coco["images"]).set_index("id")

df = pd.DataFrame(coco["annotations"])
df["class"] = df["category_id"].map(cats)
df[["x", "y", "bw", "bh"]] = pd.DataFrame(df["bbox"].tolist(), index=df.index)
df["W"] = df["image_id"].map(imgs["width"])
df["H"] = df["image_id"].map(imgs["height"])

files = os.listdir(IMG_DIR)
print("Images:", len(imgs), "| Boxes:", len(df), "| Classes:", len(cats))
print("Extensions on disk:", pd.Series([os.path.splitext(f)[1] for f in files]).value_counts().to_dict())
print("Images missing on disk:", (~imgs["file_name"].isin(files)).sum())
print("\nInstances per class:\n", df["class"].value_counts().to_string())

# short side of each box after YOLO resizes the image (longest side -> imgsz)
short = np.minimum(df["bw"], df["bh"]) / np.maximum(df["W"], df["H"])
for s in (640, 1024):
    print(f"imgsz={s}: box short side p10 = {np.percentile(short * s, 10):.0f} px, median = {np.median(short * s):.0f} px")

## 3. Clean, split and convert to YOLO format
- **Clean:** clip boxes to the image first, then remove degenerate boxes (side < 5 px). Images that end up with no boxes are left out of the split (the count is printed).
- **Split by image, not by box**, so the same drawing never appears in train and test (no leakage). Stratify on each image's *rarest* class. Keys with fewer than 10 images are merged into the largest group: with a 80/10/10 split done in two steps, each key needs at least 2 images in the temporary 20 % part, i.e. at least 10 images overall. *Caveat:* if the dataset contains several photos or scans of the same circuit, they are different images for this split, so some leakage can remain.
- **YOLO format:** one `.txt` per image, `cls xc yc w h` normalized to 0-1 (labels stay valid when YOLO resizes the image), class ids start at 0.
- The output folder is deleted first, so files from a previous run cannot end up in the wrong split.
- The crosstab at the end checks that every class appears in train, val and test.

In [ ]:
#Clip to the image, then drop tiny boxes
df["x1"], df["y1"] = df["x"].clip(lower=0), df["y"].clip(lower=0)
df["x2"], df["y2"] = np.minimum(df["x"] + df["bw"], df["W"]), np.minimum(df["y"] + df["bh"], df["H"])
df_clean = df[(df["x2"] - df["x1"] >= 5) & (df["y2"] - df["y1"] >= 5)].copy()
print(f"Removed {len(df) - len(df_clean)} degenerate boxes")

#COCO to YOLO format
cat_ids = sorted(cats)
names = [cats[c] for c in cat_ids]
df_clean["cls"] = df_clean["category_id"].map({c: i for i, c in enumerate(cat_ids)})   # COCO id -> 0..16

#Stratified split by image
class_freq = df_clean["cls"].value_counts()
img_key = df_clean.groupby("image_id")["cls"].apply(lambda s: min(s, key=lambda c: class_freq[c]))
key_counts = img_key.value_counts()
small = key_counts[key_counts < 10].index
img_key[img_key.isin(small)] = key_counts.idxmax()
print(f"Images left out (no valid box): {len(imgs) - len(img_key)}")

#Split the images
all_ids = img_key.index.values
train_ids, tmp_ids = train_test_split(all_ids, test_size=0.2, random_state=SEED, stratify=img_key[all_ids])
val_ids, test_ids  = train_test_split(tmp_ids, test_size=0.5, random_state=SEED, stratify=img_key[tmp_ids])
splits = {"train": train_ids, "val": val_ids, "test": test_ids}
print({k: len(v) for k, v in splits.items()})

#Write the files in YOLO format (xc,yc,w,h)
shutil.rmtree(OUT, ignore_errors=True)
for split, ids in splits.items():
    os.makedirs(f"{OUT}/images/{split}"); os.makedirs(f"{OUT}/labels/{split}")
    for iid in ids:
        fname = imgs.loc[iid, "file_name"]
        W, H = imgs.loc[iid, "width"], imgs.loc[iid, "height"]
        shutil.copy(os.path.join(IMG_DIR, fname), f"{OUT}/images/{split}/{fname}")
        g = df_clean[df_clean["image_id"] == iid]
        lines = [f"{c} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}"
                 for c, x1, y1, x2, y2 in zip(g["cls"], g["x1"], g["y1"], g["x2"], g["y2"])]
        open(f"{OUT}/labels/{split}/{os.path.splitext(fname)[0]}.txt", "w").write("\n".join(lines))

yaml.safe_dump({"path": OUT, "train": "images/train", "val": "images/val", "test": "images/test",
                "names": dict(enumerate(names))}, open(DATA_YAML, "w"), sort_keys=False)

#Check that every class appears in val and test
df_clean["split"] = df_clean["image_id"].map({i: s for s, ids in splits.items() for i in ids})
counts = pd.crosstab(df_clean["class"], df_clean["split"])[["train", "val", "test"]]
print(counts.sort_values("train", ascending=False))
split_df = df_clean[["image_id", "split"]].drop_duplicates()
split_df["file_name"] = split_df["image_id"].map(imgs["file_name"])
split_df.to_csv("/kaggle/working/split.csv", index=False)

## 4. Visual check of the conversion
Draw the YOLO labels back on random training images. Boxes must sit on the components and carry the right class name; if they do not, the conversion is wrong.

In [4]:
def draw_gt(img, lbl_path, color=(0, 170, 0)):
    H, W = img.shape[:2]
    for line in open(lbl_path).read().strip().splitlines():
        c, xc, yc, w, h = line.split(); c = int(c); xc, yc, w, h = map(float, (xc, yc, w, h))
        p1, p2 = (int((xc-w/2)*W), int((yc-h/2)*H)), (int((xc+w/2)*W), int((yc+h/2)*H))
        cv2.rectangle(img, p1, p2, color, 3)
        cv2.putText(img, names[c], (p1[0], max(p1[1]-6, 15)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
    return img

def label_of(path):
    return os.path.splitext(path.replace("/images/", "/labels/"))[0] + ".txt"

sample = random.sample(glob.glob(f"{OUT}/images/train/*"), 4)     # any extension (the dataset is PNG)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, path in zip(axes.ravel(), sample):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    ax.imshow(draw_gt(img, label_of(path))); ax.set_title(os.path.basename(path)); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Train
| Setting | Value | Reason |
|---|---|---|
| model | `yolo11m.pt` (COCO pretrained) | Transfer learning works well with a small dataset; the medium size keeps fine details (e.g. arrow direction) and trains in a few hours on a T4. Not benchmarked against other model sizes here. |
| imgsz | 1024 | Symbols stay large enough after resizing (see box sizes in cell 2). |
| epochs / patience | 100 / 20 | Enough to converge; training stops early if validation does not improve for 20 epochs. |
| batch | 4 | Fits a 16 GB T4 at 1024 px. |
| fliplr | 0 | Mirrored drawings / handwriting do not occur in real data. |
| degrees | 5 | Scanned or photographed drawings are often slightly tilted. |
| seed | 42 | Reproducibility. |


In [5]:
model = YOLO("yolo11m.pt")
model.train(data=DATA_YAML, imgsz=1024, epochs=100, patience=20, batch=4, device=0,
            fliplr=0.0, degrees=5, seed=SEED, workers=4,
            project=RUNS, name="yolo11m_1024", exist_ok=False, plots=True)
BEST = f"{model.trainer.save_dir}/weights/best.pt"
print(BEST)

## 6. Evaluate on the validation set
- **mAP50 and mAP50-95:**
  - mAP50: a box is correct if it overlaps the true box by at least 50 % (IoU ≥ 0.5).
  - mAP50-95: the same, averaged over stricter overlaps up to 95 %, so it also measures how tight the boxes are.
  - We validate with `conf=0.001` so the model keeps all its boxes, even the unsure ones; mAP needs the full precision-recall curve.
- **Threshold (`CONF`):** the confidence with the best mean F1 on validation. It is chosen here and reused unchanged on the test set and for predictions. The reported P and R come from Ultralytics' own best-F1 point, which can differ very slightly from `CONF`.
- **Confusion matrix:** computed in a second validation run at `CONF`. At 0.001 the matrix would be full of low-confidence false alarms.
- **`n_gt`:** number of true boxes per class in that split. A class with few boxes gives a less reliable score.

In [ ]:
model = YOLO(BEST)

def evaluate(split, conf=None):
    m = model.val(data=DATA_YAML, split=split, imgsz=1024, batch=8, conf=0.001, device=0,
                  project=RUNS, name=split, exist_ok=True)
    if conf is None:                                   # choose the threshold (validation only)
        f1 = m.box.f1_curve.mean(0)
        conf = float(np.linspace(0, 1, len(f1))[f1.argmax()])
    per_class = pd.DataFrame({
        "class": [names[c] for c in m.box.ap_class_index],
        "n_gt":  [counts.loc[names[c], split] for c in m.box.ap_class_index],
        "P": m.box.p, "R": m.box.r, "AP50": m.box.ap50, "AP50-95": m.box.ap,
    }).sort_values("AP50-95")
    overall = {"mAP50": float(m.box.map50), "mAP50-95": float(m.box.map),
               "P (best F1)": float(m.box.mp), "R (best F1)": float(m.box.mr), "conf": conf}
    m_cm = model.val(data=DATA_YAML, split=split, imgsz=1024, batch=8, conf=conf, device=0,
                     project=RUNS, name=f"{split}_cm", exist_ok=True)
    return per_class, overall, m_cm.confusion_matrix.matrix

pc_val, ov_val, cm_val = evaluate("val")
CONF = ov_val["conf"]
print(ov_val); print(pc_val.round(3).to_string(index=False))
pc_val.round(4).to_csv("/kaggle/working/val_per_class_metrics.csv", index=False)

## 7. Final evaluation on the test set (run once)
Runs only with `RUN_TEST = True`, after all decisions are fixed, and uses the threshold chosen on validation (the test set never influences any choice). `TEST_DONE.txt` records when it was run and stops a second run. The function makes two passes over the same test images (one for mAP, one for the confusion matrix), but with the same weights and no tuning in between.

In [ ]:
ov_test = None
if RUN_TEST:
    assert not os.path.exists("/kaggle/working/TEST_DONE.txt"), "Test set already evaluated"
    pc_test, ov_test, cm_test = evaluate("test", conf=CONF)
    print(ov_test); print(pc_test.round(3).to_string(index=False))
    pc_test.round(4).to_csv("/kaggle/working/test_per_class_metrics.csv", index=False)
    open("/kaggle/working/TEST_DONE.txt", "w").write(f"{datetime.datetime.now()} conf={CONF:.3f}")
else:
    print("RUN_TEST = False: test set not evaluated")

## 8. Error analysis
- **Confusion pairs:** a true class predicted as another class.
- **Missed objects:** true boxes with no detection (false negatives).
- **False alarms:** detections with no true object (false positives).

Ultralytics stores the matrix as `matrix[predicted, true]`; the last index is *background*. The analysis uses the test set in the final run and validation otherwise.

In [ ]:
split = "test" if RUN_TEST else "val"
cm = cm_test if RUN_TEST else cm_val
nc = len(names)
pairs = [(names[t], names[p], int(cm[p, t])) for t in range(nc) for p in range(nc) if p != t and cm[p, t] > 0]
print("Top class confusions (true -> predicted):")
for t, p, n in sorted(pairs, key=lambda x: -x[2])[:10]: print(f"  {t:20s} -> {p:20s} {n}")

missed = {names[t]: int(cm[nc, t]) for t in range(nc)}         # true object, no detection
false_alarm = {names[p]: int(cm[p, nc]) for p in range(nc)}    # detection, no object
print("\nMissed per class:", dict(sorted(missed.items(), key=lambda x: -x[1])[:6]))
print("False alarms per class:", dict(sorted(false_alarm.items(), key=lambda x: -x[1])[:6]))

Predictions (model colours) with the ground truth drawn on top in **green**, so missed, extra and misclassified components are visible.

In [ ]:
sample = random.sample(glob.glob(f"{OUT}/images/{split}/*"), 4)
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, path in zip(axes.ravel(), sample):
    r = model.predict(path, imgsz=1024, conf=CONF, device=0, verbose=False)[0]
    img = cv2.cvtColor(r.plot(line_width=2, font_size=10), cv2.COLOR_BGR2RGB)
    ax.imshow(draw_gt(img, label_of(path)))
    ax.set_title(f"{os.path.basename(path)}: {len(r.boxes)} detections"); ax.axis("off")
plt.tight_layout(); plt.show()

## 9. Save outputs
Saves the metrics, the threshold, the package versions and all run folders (weights, curves, confusion matrices) in one zip. Files that only exist after the test run are copied only if present, so the cell also works with `RUN_TEST = False`.

In [ ]:
json.dump({"val": ov_val, "test": ov_test}, open("/kaggle/working/results_summary.json", "w"), indent=2)
os.system("pip freeze > /kaggle/working/requirements.txt")
print(json.dumps({"val": ov_val, "test": ov_test}, indent=2))

In [ ]:
from IPython.display import FileLink
for f in ["test_per_class_metrics.csv", "val_per_class_metrics.csv", "results_summary.json",
          "requirements.txt", "split.csv", "TEST_DONE.txt"]:
    p = f"/kaggle/working/{f}"
    if os.path.exists(p):                      # test files exist only after the test run
        shutil.copy(p, f"{RUNS}/{f}")
shutil.make_archive("/kaggle/working/submission_outputs", "zip", RUNS)
FileLink("submission_outputs.zip")

## 10. Test on my own image
Qualitative check on a circuit that is not part of the dataset. It is only a visual sanity check, not a metric.

In [ ]:
from ultralytics import YOLO

WEIGHTS = "/kaggle/working/runs/yolo11m_1024/weights/best.pt"
found   = glob.glob("/kaggle/input/**/circuit0.jpg", recursive=True)
CONF    = 0.25

if not found:
    print("Own test image not attached: skipping this cell")
else:
    model = YOLO(WEIGHTS)
    r = model.predict(found[0], imgsz=1024, conf=CONF, device=0, verbose=False)[0]
    plt.figure(figsize=(14, 10))
    plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
    for c, s in zip(r.boxes.cls, r.boxes.conf):
        print(model.names[int(c)], f"{float(s):.2f}")